In [ ]:
!pip install sbi -q

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!unzip -o src.zip
!touch src/__init__.py
!ls src

In [ ]:
import os
import sys
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

from sbi.inference import NPE
from sbi.utils import BoxUniform

sys.path.append(".")

from src.ddm_simulator import simulate_ddm_dataset
from src.summary_statistics import summarize_behavior
from src.sbi_pipeline import ddm_sbi_simulator, run_trial_count_experiment
from src.plotting import (
    plot_reaction_time_distribution,
    plot_posterior_parameters,
    plot_true_vs_inferred,
    plot_uncertainty_vs_trials
)

torch.manual_seed(42)
np.random.seed(42)

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("data", exist_ok=True)

In [ ]:
true_params = {
    "drift_rate": 1.2,
    "boundary": 1.0,
    "non_decision_time": 0.25
}

param_names = ["drift_rate", "boundary", "non_decision_time"]

true_values = torch.tensor([
    true_params["drift_rate"],
    true_params["boundary"],
    true_params["non_decision_time"]
])

In [ ]:
reaction_times, choices = simulate_ddm_dataset(
    drift_rate=true_params["drift_rate"],
    boundary=true_params["boundary"],
    non_decision_time=true_params["non_decision_time"],
    n_trials=300
)

observed_data_df = pd.DataFrame({
    "reaction_time": reaction_times,
    "choice": choices
})

observed_data_df.to_csv("data/simulated_observed_data.csv", index=False)

print("Mean RT:", reaction_times.mean())
print("Choice-1 proportion:", choices.mean())

In [ ]:
plot_reaction_time_distribution(
    reaction_times,
    save_path="figures/simulated_rt_distribution.png"
)

In [ ]:
x_observed = summarize_behavior(reaction_times, choices)
x_observed_tensor = torch.tensor(x_observed, dtype=torch.float32)

x_observed

In [ ]:
prior = BoxUniform(
    low=torch.tensor([0.2, 0.5, 0.1]),
    high=torch.tensor([2.5, 2.0, 0.6])
)

In [ ]:
num_simulations = 1000

theta_samples = prior.sample((num_simulations,))
x_samples = []

for i, theta in enumerate(theta_samples):
    x = ddm_sbi_simulator(theta)
    x_samples.append(x)

    if (i + 1) % 100 == 0:
        print(f"Completed {i + 1}/{num_simulations} simulations")

x_samples = torch.stack(x_samples)

print(theta_samples.shape)
print(x_samples.shape)

In [ ]:
inference = NPE(prior=prior)

density_estimator = inference.append_simulations(
    theta_samples,
    x_samples
).train()

posterior = inference.build_posterior(density_estimator)

In [ ]:
posterior_samples = posterior.sample(
    (2000,),
    x=x_observed_tensor
)

posterior_mean = posterior_samples.mean(dim=0)
posterior_std = posterior_samples.std(dim=0)

In [ ]:
results_df = pd.DataFrame({
    "parameter": param_names,
    "true_value": true_values.numpy(),
    "posterior_mean": posterior_mean.detach().numpy(),
    "posterior_std": posterior_std.detach().numpy()
})

results_df.to_csv("results/parameter_recovery_results.csv", index=False)

results_df

In [ ]:
plot_posterior_parameters(
    posterior_samples=posterior_samples,
    true_values=true_values,
    posterior_mean=posterior_mean,
    param_names=param_names,
    save_path="figures/posterior_parameters.png"
)

In [ ]:
plot_true_vs_inferred(
    results_df=results_df,
    param_names=param_names,
    save_path="figures/true_vs_inferred_parameters.png"
)

In [ ]:
trial_counts = [25, 50, 100, 200, 500]

uncertainty_results = []

for n_trials in trial_counts:
    print(f"Running inference for {n_trials} observed trials...")

    mean_temp, std_temp = run_trial_count_experiment(
        posterior=posterior,
        true_params=true_params,
        n_observed_trials=n_trials,
        n_posterior_samples=1000
    )

    for i, name in enumerate(param_names):
        uncertainty_results.append({
            "n_trials": n_trials,
            "parameter": name,
            "posterior_mean": mean_temp[i].item(),
            "posterior_std": std_temp[i].item()
        })

uncertainty_df = pd.DataFrame(uncertainty_results)

uncertainty_df.to_csv("results/uncertainty_vs_trials.csv", index=False)

uncertainty_df

In [ ]:
plot_uncertainty_vs_trials(
    uncertainty_df=uncertainty_df,
    param_names=param_names,
    save_path="figures/uncertainty_vs_trials.png"
)

In [ ]:
!ls figures
!ls results
!ls data

In [ ]:
from google.colab import files

files.download("figures/simulated_rt_distribution.png")
files.download("figures/posterior_parameters.png")
files.download("figures/true_vs_inferred_parameters.png")
files.download("figures/uncertainty_vs_trials.png")
files.download("results/parameter_recovery_results.csv")
files.download("results/uncertainty_vs_trials.csv")
files.download("data/simulated_observed_data.csv")